# H1.Z16 Isolation Forest 이상탐지

## 01. 목적

`H1.Z16` 계량기의 1시간 전기 measurement를 이용해 Isolation Forest 기반 비지도 이상 후보 탐지를 수행한다. 결과는 고장 확정 label이 아니라 검토 대상 후보로 해석한다.

## 02. 데이터 출처

- 시계열 값: TimescaleDB hypertable `ems.cr_measurement_1h`
- 적재/전처리 상태: `ems.full_source_file`
- 대상 계량기: `H1.Z16`
- 계통 해석: `central_cooling` pilot 기준

## 03. Measurement 사전

`H1.Z16` 전기 계량기에서 사용하는 주요 measurement의 의미는 다음과 같다. 한글명은 분석 해석용 명칭이며, 단위와 영문 설명은 DB의 `ems.full_measurement_definition` 기준이다.

| measurement | 한글명 | 단위 | 해석 |
|---|---|---:|---|
| `P` | 유효전력 | W | 전체 3상 기준 실제 일을 하는 전력. 부하 패턴과 이상탐지의 핵심 변수. |
| `P1`, `P2`, `P3` | 상별 유효전력 | W | 1상, 2상, 3상별 유효전력. 상별 부하 불균형 확인에 사용. |
| `W` | 유효전력량 | kWh | 누적 또는 구간 유효 에너지. 모델 입력에서는 차분값 `delta_W`를 우선 사용. |
| `W_in` | 유입 유효전력량 | kWh | 계통에서 유입된 유효 에너지. 소비 방향 에너지 해석에 사용. |
| `W_out` | 유출 유효전력량 | kWh | 계통으로 유출된 유효 에너지. 발전/역방향 흐름 확인에 사용. |
| `I1`, `I2`, `I3` | 상별 전류 | A | 1상, 2상, 3상 전류. `I_imbalance` 파생 feature 생성에 사용. |
| `U1`, `U2`, `U3` | 상별 전압 | V | 1상, 2상, 3상 전압. 전압 안정성과 `U_imbalance` 확인에 사용. |
| `PF` | 역률 | - | 전체 역률. 유효전력 대비 전력 사용 효율과 무효전력 영향을 설명. |
| `PF1`, `PF2`, `PF3` | 상별 역률 | - | 1상, 2상, 3상별 역률. 상별 품질 차이 확인에 사용. |
| `Q` | 무효전력 | var | 실제 일을 하지 않지만 전자기장 형성에 필요한 전력 성분. `Q_ratio` 생성에 사용. |
| `WQ` | 무효전력량 | kvarh | 누적 또는 구간 무효 에너지. 모델 입력에서는 차분값 `delta_WQ`를 우선 사용. |
| `WQ_in` | 유입 무효전력량 | kvarh | 유입 방향 무효 에너지. |
| `WQ_out` | 유출 무효전력량 | kvarh | 유출 방향 무효 에너지. |
| `f` | 계통 주파수 | Hz | 전력망 주파수. 개별 설비 분류보다는 전력 품질/상태 feature로 사용. |


## 04. 모델링 계획

1. `H1.Z16`의 1시간 데이터를 조회하고 wide format으로 변환한다.
2. DB 적재 상태와 시간축을 간단히 확인한다.
3. 유효전력, 전류, 전압, 역률, 무효전력, 주파수 기반 feature를 생성한다.
4. median imputation과 robust scaling 후 Isolation Forest를 학습한다.
5. anomaly score와 상위 후보 시점을 검토한다.


In [ ]:
# 01. 기본 설정 및 dependency 확인
from pathlib import Path
import os
import sys
import warnings

warnings.filterwarnings('default')

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'HERMES.md').exists() and (PROJECT_ROOT.parent / 'HERMES.md').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'
OUTPUT_DIR = PROJECT_ROOT / 'outputs' / 'tables' / 'h1_z16_iforest'

METER_URN = 'H1.Z16'
EQUIPMENT_GROUP = 'central_cooling'
METER_ROLE = 'consumption'
RESOLUTION = '1h'
SOURCE_TABLE = 'ems.cr_measurement_1h'

QUICK_RUN = os.getenv('EMS_NOTEBOOK_QUICK', '0') == '1'
START_TS = '2023-01-01' if QUICK_RUN else None
END_TS = '2023-03-01' if QUICK_RUN else None
SAVE_OUTPUTS = os.getenv('EMS_SAVE_NOTEBOOK_OUTPUTS', '0') == '1'

PROJECT_ROOT: c:\Arch Linux\EMS\notebooks
METER_URN: H1.Z16
SOURCE_TABLE: ems.cr_measurement_1h
QUICK_RUN: False START_TS: None END_TS: None
SAVE_OUTPUTS: False


In [4]:
# 02. Python package imports
required = {
    'pandas': 'pandas',
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
    'koreanize_matplotlib': 'koreanize-matplotlib',
    'seaborn': 'seaborn',
    'psycopg': 'psycopg[binary]',
    'sklearn': 'scikit-learn',
}
missing = []
for module_name, package_name in required.items():
    try:
        __import__(module_name)
    except ImportError:
        missing.append(package_name)

if missing:
    raise ImportError(
        '필수 패키지가 없습니다: ' + ', '.join(missing) +
        '\n예: python -m pip install ' + ' '.join(missing)
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import psycopg
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['axes.grid'] = True
sns.set_theme(style='whitegrid')
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [6]:
# 03. DB 접속 정보 로드
def load_env_file(env_path: Path) -> None:
    if not env_path.exists():
        return
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if not line or line.startswith('#') or '=' not in line:
            continue
        key, value = line.split('=', 1)
        key = key.strip()
        value = value.strip().strip('"').strip("'")
        os.environ.setdefault(key, value)

load_env_file(PROJECT_ROOT / '.env')

required_env = ['DB_HOST', 'DB_PORT', 'DB_NAME', 'DB_USER', 'DB_PASSWORD']
missing_env = [key for key in required_env if not os.getenv(key)]
if missing_env:
    raise RuntimeError(f'DB 접속 환경변수가 없습니다: {missing_env}')

In [7]:
# 04. DB helper
CONNECT_KWARGS = {
    'host': os.environ['DB_HOST'],
    'port': int(os.environ.get('DB_PORT', '5432')),
    'dbname': os.environ['DB_NAME'],
    'user': os.environ['DB_USER'],
    'password': os.environ['DB_PASSWORD'],
}

def query_df(sql: str, params=None) -> pd.DataFrame:
    with psycopg.connect(**CONNECT_KWARGS) as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            cols = [desc.name for desc in cur.description]
            rows = cur.fetchall()
    return pd.DataFrame(rows, columns=cols)

# 접속 및 대상 table 확인
relation_check = query_df("""
SELECT to_regclass(%s) AS relation_name
""", (SOURCE_TABLE,))
relation_check

,relation_name
0,ems.cr_measurement_1h


In [8]:
# 05. 대상 계량기 measurement coverage 확인
coverage_sql = f"""
WITH measurement_map AS (
    SELECT *
    FROM (VALUES
        ('P', '유효전력', '전체 3상 기준 실제 일을 하는 전력'),
        ('P1', '1상 유효전력', '1상 기준 유효전력'),
        ('P2', '2상 유효전력', '2상 기준 유효전력'),
        ('P3', '3상 유효전력', '3상 기준 유효전력'),
        ('W', '유효전력량', '유효전력의 에너지 누적 또는 구간값'),
        ('W_in', '유입 유효전력량', '유입 방향 유효 에너지'),
        ('W_out', '유출 유효전력량', '유출 방향 유효 에너지'),
        ('I1', '1상 전류', '1상 전류'),
        ('I2', '2상 전류', '2상 전류'),
        ('I3', '3상 전류', '3상 전류'),
        ('U1', '1상 전압', '1상 전압'),
        ('U2', '2상 전압', '2상 전압'),
        ('U3', '3상 전압', '3상 전압'),
        ('PF', '역률', '전체 역률'),
        ('PF1', '1상 역률', '1상 역률'),
        ('PF2', '2상 역률', '2상 역률'),
        ('PF3', '3상 역률', '3상 역률'),
        ('Q', '무효전력', '무효전력'),
        ('WQ', '무효전력량', '무효전력의 에너지 누적 또는 구간값'),
        ('WQ_in', '유입 무효전력량', '유입 방향 무효 에너지'),
        ('WQ_out', '유출 무효전력량', '유출 방향 무효 에너지'),
        ('f', '계통 주파수', '전력망 주파수')
    ) AS t(measurement, measurement_ko, analysis_note)
)
SELECT
    m.meter_urn,
    m.measurement,
    mm.measurement_ko,
    COALESCE(d.unit, '') AS unit,
    d.description AS description_en,
    mm.analysis_note,
    count(*) AS rows,
    min(m.ts) AS min_ts,
    max(m.ts) AS max_ts,
    count(*) FILTER (WHERE m.value IS NULL) AS null_rows,
    count(*) FILTER (WHERE m.value::text IN ('NaN', 'Infinity', '-Infinity')) AS non_finite_rows
FROM {SOURCE_TABLE} AS m
LEFT JOIN ems.full_measurement_definition AS d
    ON d.measurement = m.measurement
LEFT JOIN measurement_map AS mm
    ON mm.measurement = m.measurement
WHERE m.meter_urn = %s
GROUP BY
    m.meter_urn,
    m.measurement,
    mm.measurement_ko,
    d.unit,
    d.description,
    mm.analysis_note
ORDER BY m.measurement
"""
coverage = query_df(coverage_sql, (METER_URN,))
coverage


,meter_urn,measurement,rows,min_ts,max_ts,null_rows,non_finite_rows
0,H1.Z16,I1,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
1,H1.Z16,I2,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
2,H1.Z16,I3,52523,2018-01-03 12:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
3,H1.Z16,P,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
4,H1.Z16,P1,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
5,H1.Z16,P2,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
6,H1.Z16,P3,52523,2018-01-03 12:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
7,H1.Z16,PF,52583,2018-01-01 00:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
8,H1.Z16,PF1,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0
9,H1.Z16,PF2,52584,2017-12-31 23:00:00+00:00,2023-12-31 22:00:00+00:00,0,0


In [9]:
# 06. 분석 대상 measurement 선정
candidate_measurements = [
    'P', 'P1', 'P2', 'P3',
    'W', 'W_in', 'W_out',
    'I1', 'I2', 'I3',
    'U1', 'U2', 'U3',
    'PF', 'PF1', 'PF2', 'PF3',
    'Q', 'WQ', 'WQ_in', 'WQ_out',
    'f',
]

available_measurements = coverage['measurement'].tolist()
measurements = [m for m in candidate_measurements if m in available_measurements]
missing_measurements = [m for m in candidate_measurements if m not in available_measurements]

print('available selected measurements:', measurements)
print('missing selected measurements:', missing_measurements)
print('measurement count:', len(measurements))

available selected measurements: ['P', 'P1', 'P2', 'P3', 'W', 'W_in', 'W_out', 'I1', 'I2', 'I3', 'U1', 'U2', 'U3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'WQ', 'WQ_in', 'WQ_out', 'f']
missing selected measurements: []
measurement count: 22


In [10]:
# 07. H1.Z16 1h 데이터 조회
where_clauses = ["meter_urn = %s", "measurement = ANY(%s)"]
params = [METER_URN, measurements]
if START_TS is not None:
    where_clauses.append('ts >= %s')
    params.append(START_TS)
if END_TS is not None:
    where_clauses.append('ts < %s')
    params.append(END_TS)

raw_sql = f"""
SELECT ts, measurement, value
FROM {SOURCE_TABLE}
WHERE {' AND '.join(where_clauses)}
ORDER BY ts, measurement
"""
raw = query_df(raw_sql, tuple(params))
raw['ts'] = pd.to_datetime(raw['ts'], utc=True)
raw['value'] = pd.to_numeric(raw['value'], errors='coerce')

print('raw shape:', raw.shape)
print('time range:', raw['ts'].min(), '→', raw['ts'].max())
raw.head()

raw shape: (1155286, 3)
time range: 2017-12-31 23:00:00+00:00 → 2024-01-01 00:00:00+00:00


,ts,measurement,value
0,2017-12-31 23:00:00+00:00,I1,1.107802
1,2017-12-31 23:00:00+00:00,I2,1.110238
2,2017-12-31 23:00:00+00:00,P,435.787833
3,2017-12-31 23:00:00+00:00,P1,230.971167
4,2017-12-31 23:00:00+00:00,P2,204.843500


In [11]:
# 08. Long format → Wide format
wide = (
    raw.pivot_table(index='ts', columns='measurement', values='value', aggfunc='mean')
    .sort_index()
)

if len(wide) > 0:
    full_index = pd.date_range(wide.index.min(), wide.index.max(), freq='1h', tz='UTC')
    wide = wide.reindex(full_index)
    wide.index.name = 'ts'

print('wide shape:', wide.shape)
print('columns:', wide.columns.tolist())
wide.head()

wide shape: (52586, 22)
columns: ['I1', 'I2', 'I3', 'P', 'P1', 'P2', 'P3', 'PF', 'PF1', 'PF2', 'PF3', 'Q', 'U1', 'U2', 'U3', 'W', 'WQ', 'WQ_in', 'WQ_out', 'W_in', 'W_out', 'f']


measurement,I1,I2,I3,P,P1,P2,P3,PF,PF1,PF2,...,U1,U2,U3,W,WQ,WQ_in,WQ_out,W_in,W_out,f
ts,,,,,,,,,,,,,,,,,,,,,
2017-12-31 23:00:00+00:00,1.107802,1.110238,NaN,435.787833,230.971167,204.843500,NaN,NaN,0.913802,0.807576,...,228.380833,228.296667,228.641667,NaN,NaN,NaN,NaN,NaN,NaN,50.035167
2018-01-01 00:00:00+00:00,1.101292,1.108417,NaN,432.206000,228.914833,203.272500,NaN,0.994333,0.913500,0.807892,...,227.332500,227.390000,227.714167,1286.01,135.81,135.81,NaN,1286.01,NaN,50.042500
2018-01-01 01:00:00+00:00,1.107000,1.110000,NaN,435.176167,230.607000,204.539833,NaN,0.994000,0.913904,0.807642,...,228.189167,228.084167,228.571667,1286.44,135.85,135.85,NaN,1286.44,NaN,50.044667
2018-01-01 02:00:00+00:00,1.106333,1.109083,NaN,434.345000,230.063250,204.217167,NaN,0.994000,0.913687,0.807642,...,227.923333,227.957500,228.361667,1286.88,135.90,135.90,NaN,1286.88,NaN,50.031000
2018-01-01 03:00:00+00:00,1.106000,1.109333,NaN,434.194167,230.008917,204.162083,NaN,0.994000,0.913617,0.807950,...,227.808333,227.814167,228.180000,1287.31,135.95,135.95,NaN,1287.31,NaN,50.030333



## 05. 전처리 상태 확인

Isolation Forest를 적용하기 전에 현재 DB 데이터가 어느 정도 전처리되어 있는지 확인한다. 본 노트북에서는 원천 CSV를 직접 전처리하지 않고, 이미 DB에 적재된 `corrected_resampled` 1시간 mart를 분석 입력으로 사용한다.

확인 항목은 다음과 같다.

1. `full_source_file` 기준 처리 단계, 해상도, 품질 counter
2. `cr_measurement_1h` 기준 row 수, measurement 수, 시간 범위, null/non-finite 여부
3. wide table 기준 시간 간격, 중복 timestamp, 결측률
4. 누적 전력량 계열의 차분 음수 여부
5. Isolation Forest 입력 전 추가 전처리 필요 항목


In [12]:
# 09. full_source_file 기준 H1.Z16 전처리/적재 상태 확인
preprocess_state_sql = """
SELECT
    processing_level,
    resolution_code,
    count(*) AS source_files,
    count(DISTINCT measurement) AS measurements,
    sum(csv_rows) AS csv_rows,
    sum(inserted_rows) AS inserted_rows,
    sum(conflict_rows) AS conflict_rows,
    sum(null_value_rows) AS null_value_rows,
    sum(invalid_value_rows) AS invalid_value_rows,
    sum(invalid_ts_rows) AS invalid_ts_rows,
    sum(non_finite_rows) AS non_finite_rows,
    min(started_at) AS first_load_started,
    max(finished_at) AS last_load_finished
FROM ems.full_source_file
WHERE meter_urn = %s
GROUP BY processing_level, resolution_code
ORDER BY processing_level, resolution_code
"""
preprocess_state = query_df(preprocess_state_sql, (METER_URN,))
preprocess_state

,processing_level,resolution_code,source_files,measurements,csv_rows,inserted_rows,conflict_rows,null_value_rows,invalid_value_rows,invalid_ts_rows,non_finite_rows,first_load_started,last_load_finished
0,corrected_resampled,15min,22,22,4621135,4621135,0,0,0,0,0,2026-05-06 04:31:18.839116+00:00,2026-05-06 04:33:40.974955+00:00
1,corrected_resampled,1h,22,22,1155286,1155286,0,0,0,0,0,2026-05-06 03:36:55.255270+00:00,2026-05-06 03:37:28.228716+00:00
2,corrected_resampled,1min,22,22,69316987,69316987,0,0,0,0,0,2026-05-07 09:27:41.345003+00:00,2026-05-07 11:59:56.369754+00:00


In [13]:
# 10. 현재 분석 입력 table(cr_measurement_1h) 상태 확인
cr_state_sql = f"""
SELECT
    %s::text AS source_table,
    count(*) AS rows,
    count(DISTINCT measurement) AS measurements,
    min(ts) AS min_ts,
    max(ts) AS max_ts,
    count(*) FILTER (WHERE value IS NULL) AS null_rows,
    count(*) FILTER (WHERE value::text IN ('NaN', 'Infinity', '-Infinity')) AS non_finite_rows
FROM {SOURCE_TABLE}
WHERE meter_urn = %s
"""
cr_state = query_df(cr_state_sql, (SOURCE_TABLE, METER_URN))
cr_state

,source_table,rows,measurements,min_ts,max_ts,null_rows,non_finite_rows
0,ems.cr_measurement_1h,1155286,22,2017-12-31 23:00:00+00:00,2024-01-01 00:00:00+00:00,0,0


In [14]:
# 11. wide table 기준 시간축/결측 상태 확인
if len(wide) == 0:
    raise RuntimeError('wide table is empty')

expected_index = pd.date_range(wide.index.min(), wide.index.max(), freq='1h', tz='UTC')
missing_timestamps = expected_index.difference(wide.index)
extra_timestamps = wide.index.difference(expected_index)

time_grid_summary = pd.DataFrame([
    {
        'start_ts': wide.index.min(),
        'end_ts': wide.index.max(),
        'actual_rows': len(wide),
        'expected_hourly_rows': len(expected_index),
        'missing_timestamps': len(missing_timestamps),
        'extra_timestamps': len(extra_timestamps),
        'duplicated_timestamps': int(wide.index.duplicated().sum()),
        'columns': len(wide.columns),
        'max_column_null_ratio': float(wide.isna().mean().max()),
    }
])
time_grid_summary

,start_ts,end_ts,actual_rows,expected_hourly_rows,missing_timestamps,extra_timestamps,duplicated_timestamps,columns,max_column_null_ratio
0,2017-12-31 23:00:00+00:00,2024-01-01 00:00:00+00:00,52586,52586,0,0,0,22,0.01314


In [16]:
# 12. 누적 전력량 계열 차분 점검
energy_cols = [c for c in ['W', 'W_in', 'W_out', 'WQ', 'WQ_in', 'WQ_out'] if c in wide.columns]
energy_diff_checks = []
for col in energy_cols:
    diff = wide[col].diff()
    energy_diff_checks.append({
        'column': col,
        'non_null_diff_rows': int(diff.notna().sum()),
        'negative_diff_rows': int((diff < 0).sum()),
        'zero_diff_rows': int((diff == 0).sum()),
        'min_diff': float(diff.min(skipna=True)) if diff.notna().any() else np.nan,
        'median_diff': float(diff.median(skipna=True)) if diff.notna().any() else np.nan,
        'max_diff': float(diff.max(skipna=True)) if diff.notna().any() else np.nan,
    })
energy_diff_check = pd.DataFrame(energy_diff_checks)
energy_diff_check

,column,non_null_diff_rows,negative_diff_rows,zero_diff_rows,min_diff,median_diff,max_diff
0,W,52584,0,11,0.0,0.44,38.09
1,W_in,52582,0,10,0.0,0.44,38.09
2,W_out,51894,0,51894,0.0,0.00,0.00
3,WQ,52584,0,14,0.0,0.05,25.65
4,WQ_in,52582,0,13,0.0,0.05,25.65
5,WQ_out,51894,0,51894,0.0,0.00,0.00


## 06. Isolation Forest 입력 feature 생성

Isolation Forest에는 raw measurement와 해석 가능한 파생 feature를 함께 사용한다. 누적 전력량 계열은 그대로 사용하지 않고 차분 feature를 우선 사용한다. 차분값이 음수로 크게 나타나는 경우는 누적 reset 또는 source 특성 후보로 보고 결측 처리한다.


In [ ]:
# 13. Feature engineering helpers
def safe_diff(series: pd.Series, negative_to_nan: bool = True) -> pd.Series:
    diff = series.diff()
    if negative_to_nan:
        # 누적형 meter reset 또는 source 특성 후보를 모델 feature에서 제외한다.
        diff = diff.mask(diff < 0)
    return diff

def phase_imbalance(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    existing = [c for c in cols if c in df.columns]
    if len(existing) < 2:
        return pd.Series(np.nan, index=df.index)
    vals = df[existing]
    mean_abs = vals.mean(axis=1).abs().replace(0, np.nan)
    return (vals.max(axis=1) - vals.min(axis=1)).abs() / mean_abs

def safe_ratio(num: pd.Series, den: pd.Series) -> pd.Series:
    return num / den.abs().replace(0, np.nan)


In [ ]:
# 14. Feature table 구성
features = pd.DataFrame(index=wide.index)

# Raw core features
raw_feature_cols = [
    'P', 'P1', 'P2', 'P3',
    'I1', 'I2', 'I3',
    'U1', 'U2', 'U3',
    'PF', 'PF1', 'PF2', 'PF3',
    'Q', 'f',
]
for col in raw_feature_cols:
    if col in wide.columns:
        features[col] = wide[col]

# Energy deltas
for col in ['W', 'W_in', 'W_out', 'WQ', 'WQ_in', 'WQ_out']:
    if col in wide.columns:
        features[f'delta_{col}'] = safe_diff(wide[col])

# Phase imbalance and ratio features
features['I_imbalance'] = phase_imbalance(wide, ['I1', 'I2', 'I3'])
features['U_imbalance'] = phase_imbalance(wide, ['U1', 'U2', 'U3'])
if 'Q' in wide.columns and 'P' in wide.columns:
    features['Q_ratio'] = safe_ratio(wide['Q'], wide['P'])
if 'WQ' in wide.columns and 'W' in wide.columns:
    features['WQ_to_W_ratio'] = safe_ratio(safe_diff(wide['WQ']), safe_diff(wide['W']))

# P lag/rolling features
if 'P' in wide.columns:
    features['P_lag_1h'] = wide['P'].shift(1)
    features['P_lag_24h'] = wide['P'].shift(24)
    features['P_roll_mean_24h'] = wide['P'].rolling(24, min_periods=6).mean()
    features['P_roll_std_24h'] = wide['P'].rolling(24, min_periods=6).std()

# Time context, cyclical encoding
hour = wide.index.hour
weekday = wide.index.dayofweek
features['hour_sin'] = np.sin(2 * np.pi * hour / 24)
features['hour_cos'] = np.cos(2 * np.pi * hour / 24)
features['weekday_sin'] = np.sin(2 * np.pi * weekday / 7)
features['weekday_cos'] = np.cos(2 * np.pi * weekday / 7)
features['is_weekend'] = (weekday >= 5).astype(int)

features = features.replace([np.inf, -np.inf], np.nan)
feature_null_ratio = features.isna().mean().sort_values(ascending=False)
print('features shape:', features.shape)
feature_null_ratio.to_frame('null_ratio').T


In [ ]:
# 15. 모델 입력 feature 선택
# 결측률이 너무 높은 feature는 제외한다. 초기 기준은 20% 미만 결측이다.
MAX_NULL_RATIO = 0.20
selected_features = [c for c in features.columns if features[c].isna().mean() <= MAX_NULL_RATIO]

# 모든 값이 동일한 feature 제외
selected_features = [c for c in selected_features if features[c].nunique(dropna=True) > 1]

X = features[selected_features].copy()
print('selected feature count:', len(selected_features))
print(selected_features)
X.describe().T.head(30)


## 07. Isolation Forest 모델링

현재 데이터에는 정상/이상 label이 없으므로, Isolation Forest는 `정상/이상 분류기`가 아니라 `이상 후보 점수화 모델`로 사용한다.

- `contamination=0.01`: 전체 시점 중 약 1%를 검토 후보로 산출
- `n_estimators=300`: 초기 baseline 안정성 확보
- scaling/imputation: median imputation 후 robust scaling 적용


In [ ]:
# 16. Isolation Forest 학습
if X.empty or len(selected_features) == 0:
    raise RuntimeError('Isolation Forest에 사용할 feature가 없습니다.')

iso_params = {
    'n_estimators': 300,
    'max_samples': 'auto',
    'contamination': 0.01,
    'max_features': 1.0,
    'bootstrap': False,
    'random_state': 42,
    'n_jobs': -1,
}

model = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler()),
    ('isolation_forest', IsolationForest(**iso_params)),
])

model.fit(X)

# sklearn IsolationForest: decision_function이 낮을수록 이상, score_samples도 낮을수록 이상.
decision_score = model.decision_function(X)
anomaly_score = -decision_score
prediction = model.predict(X)  # -1 anomaly, 1 normal

result = pd.DataFrame(index=X.index)
result['meter_urn'] = METER_URN
result['equipment_group'] = EQUIPMENT_GROUP
result['iforest_decision_score'] = decision_score
result['anomaly_score'] = anomaly_score
result['is_anomaly_candidate'] = prediction == -1

print('model params:', iso_params)
print('rows scored:', len(result))
print('candidate count:', int(result['is_anomaly_candidate'].sum()))
print('candidate ratio:', result['is_anomaly_candidate'].mean())
result.head()


In [ ]:
# 17. Top anomaly candidates
review_cols = [c for c in ['P', 'Q', 'PF', 'I_imbalance', 'U_imbalance', 'Q_ratio', 'delta_W', 'delta_WQ', 'f'] if c in features.columns]
review = result.join(wide[[c for c in ['P', 'W', 'Q', 'PF', 'f'] if c in wide.columns]], how='left')
review = review.join(features[[c for c in review_cols if c not in review.columns]], how='left')

top_candidates = (
    review.sort_values('anomaly_score', ascending=False)
    .head(50)
    .reset_index()
    .rename(columns={'index': 'ts'})
)

top_candidates.head(20)


In [ ]:
# 18. Anomaly score 분포 및 P 시계열 표시
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=False)
axes[0].hist(result['anomaly_score'], bins=80)
axes[0].set_title('Isolation Forest anomaly score distribution')
axes[0].set_xlabel('anomaly_score')
axes[0].set_ylabel('count')

if 'P' in wide.columns:
    axes[1].plot(wide.index, wide['P'], linewidth=0.7, label='P')
    candidate_idx = result.index[result['is_anomaly_candidate']]
    axes[1].scatter(candidate_idx, wide.loc[candidate_idx, 'P'], s=25, color='red', label='candidate', zorder=3)
    axes[1].set_title('P with Isolation Forest anomaly candidates')
    axes[1].legend()
else:
    axes[1].plot(result.index, result['anomaly_score'], linewidth=0.7, label='anomaly_score')
    axes[1].set_title('Anomaly score over time')
    axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
# 19. 후보 시점 주변 feature 검토 함수
def inspect_candidate(candidate_rank: int = 0, window_hours: int = 24):
    if len(top_candidates) == 0:
        print('No candidates')
        return None
    ts = pd.to_datetime(top_candidates.loc[candidate_rank, 'ts'])
    start = ts - pd.Timedelta(hours=window_hours)
    end = ts + pd.Timedelta(hours=window_hours)
    cols = [c for c in ['P', 'Q', 'PF', 'f', 'I_imbalance', 'U_imbalance', 'Q_ratio'] if c in review.columns]
    subset = review.loc[(review.index >= start) & (review.index <= end), cols + ['anomaly_score', 'is_anomaly_candidate']]

    n = len(cols) + 1
    fig, axes = plt.subplots(n, 1, figsize=(14, max(4, 2.2 * n)), sharex=True)
    if n == 1:
        axes = [axes]
    for ax, col in zip(axes, cols):
        ax.plot(subset.index, subset[col], linewidth=0.8)
        ax.axvline(ts, color='red', linestyle='--', linewidth=1)
        ax.set_title(col)
    axes[-1].plot(subset.index, subset['anomaly_score'], linewidth=0.8)
    axes[-1].axvline(ts, color='red', linestyle='--', linewidth=1)
    axes[-1].set_title('anomaly_score')
    plt.tight_layout()
    plt.show()
    return subset

inspect_candidate(candidate_rank=0, window_hours=24)


In [ ]:
# 20. 시간대/요일별 후보 분포
candidate_only = result[result['is_anomaly_candidate']].copy()
if len(candidate_only) > 0:
    candidate_only['hour'] = candidate_only.index.hour
    candidate_only['weekday'] = candidate_only.index.dayofweek
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    candidate_only['hour'].value_counts().sort_index().plot(kind='bar', ax=axes[0])
    axes[0].set_title('Candidate count by hour')
    axes[0].set_xlabel('hour')
    candidate_only['weekday'].value_counts().sort_index().plot(kind='bar', ax=axes[1])
    axes[1].set_title('Candidate count by weekday')
    axes[1].set_xlabel('weekday (Mon=0)')
    plt.tight_layout()
    plt.show()
else:
    print('No anomaly candidates')


## 08. 결과 저장 및 후속 검토

기본값에서는 notebook 실행만으로 산출 파일을 저장하지 않는다. 후보 표를 저장하려면 환경변수 `EMS_SAVE_NOTEBOOK_OUTPUTS=1`로 실행하거나 아래 `SAVE_OUTPUTS` 값을 `True`로 변경한다.

후속 검토 기준은 다음과 같다.

1. 상위 후보가 단순 결측/flatline/누적값 reset인지 확인한다.
2. `P`, `PF`, `Q_ratio`, `I_imbalance`, `U_imbalance` 중 어떤 feature가 후보의 원인인지 확인한다.
3. 같은 `central_cooling` 계통의 `H1.Z11`, `H1.Z12`, `H1.Z24`, `H1.Z25`와 비교한다.
4. 열 계통 `V.K21` 및 외기온도 `WeatherStation.Weather.Ta`와 연결해 냉방 부하 증가인지 이상 후보인지 구분한다.
5. 사람이 검토한 결과를 별도 table로 축적하면 이후 지도학습 label로 발전시킬 수 있다.


In [ ]:
# 21. 선택적 결과 저장
if SAVE_OUTPUTS:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    candidate_path = OUTPUT_DIR / 'h1_z16_iforest_top_candidates.csv'
    scored_path = OUTPUT_DIR / 'h1_z16_iforest_scored_timeseries.csv'
    top_candidates.to_csv(candidate_path, index=False, encoding='utf-8-sig')
    result.reset_index().rename(columns={'index': 'ts'}).to_csv(scored_path, index=False, encoding='utf-8-sig')
    print('saved:', candidate_path)
    print('saved:', scored_path)
else:
    print('SAVE_OUTPUTS=False: 산출 파일을 저장하지 않았습니다.')


## 09. 해석 시 유의 사항

- Isolation Forest 결과는 이상 확정이 아니라 검토 후보이다.
- `contamination=0.01`은 실제 이상 비율을 의미하지 않는다. 검토 가능한 후보 수를 만들기 위한 초기 가정이다.
- 모델은 시간 순서를 직접 학습하지 않는다. 시간 의존성은 lag/rolling/time feature로 보완한다.
- `W`, `WQ` 계열의 차분 feature는 누적형이라는 가정이 포함된다. 음수 차분이 많으면 source semantics를 별도로 확인해야 한다.
- 후속 단계에서는 `H1.Z16` 단일 meter 결과를 같은 `central_cooling` group과 열/기상 context에 연결해야 한다.
